In [1]:
import json
from collections import Counter

In [2]:
ann_code = "C"
ann_path = f"assign/annotator_{ann_code}.txt"

with open(ann_path) as in_file:
    lines = []
    for line in in_file:
        lines.append(line)
    lines = lines[1:]
    lines = [line.split(",")[0] for line in lines]

print(f"lines: {len(lines)}")

lines: 51


In [3]:
counter = Counter(lines)
duplicates = [item for item, count in counter.items() if count > 1]
duplicates

[]

In [4]:
# "18--mixtral-8x22b", "67--mixtral-8x22b", "71--mixtral-8x22b"
dpr_id_to_dpr = dict()
SPLIT="test"

for DATA_NAME in ['TATQA', 'HybridQA', 'ConvFinQA']:
    DPR_PATH=f"../benchmark_data/{DATA_NAME}/{DATA_NAME}_{SPLIT}.jsonl"
    with open(DPR_PATH) as in_file:
        for line in in_file:
            data = json.loads(line)
            dpr_id_to_dpr[data['dpr_id']] = data

print(f"dpr_id_to_dpr: {len(dpr_id_to_dpr)}")

dpr_id_to_dpr: 2054


In [5]:
corpus_data = []

for DATA_NAME in ['TATQA', 'HybridQA', 'ConvFinQA']:
    CORPUS_PATH=f"../benchmark_data/{DATA_NAME}/{DATA_NAME}_corpus.json"
    with open(CORPUS_PATH) as in_file:
        corpus_data_t = json.load(in_file)
        corpus_data += corpus_data_t

print(f"corpus_data: {len(corpus_data)}")

corpus_data: 20111


In [6]:
table_id_to_metadata = dict()
for table_data in corpus_data:
    table_id = table_data['table']['uid']
    table_title = table_data['table']['title']
    table_headers = table_data['table']['header']
    table_id_to_metadata[table_id] = [table_id, table_title, table_headers]

print(f"table_id_to_metadata: {len(table_id_to_metadata)}")

table_id_to_metadata: 20111


In [24]:
dpr_id = lines[0]

In [9]:
for dpr_id in lines:
    dpr_text = dpr_id_to_dpr[dpr_id]['DPR']
    gt_tables = dpr_id_to_dpr[dpr_id]['ground_truth']['table']
    gt_tables_metadata = []
    for gt_table in gt_tables:
        gt_tables_metadata.append(table_id_to_metadata[gt_table])
    create_dpr_excel(dpr_text, gt_tables_metadata, output_filename=f'{ann_code}__{dpr_id}.xlsx')

Excel file saved as: C__12--qwen-2-5-72b.xlsx
Excel file saved as: C__57--llama-3-3-70b.xlsx
Excel file saved as: C__102--mixtral-8x22b.xlsx
Excel file saved as: C__393--DeepSeek-V3.xlsx
Excel file saved as: C__253--llama-3-3-70b.xlsx
Excel file saved as: C__175--llama-3-3-70b.xlsx
Excel file saved as: C__102--llama-3-3-70b.xlsx
Excel file saved as: C__339--mixtral-8x22b.xlsx
Excel file saved as: C__39--qwen-2-5-72b.xlsx
Excel file saved as: C__180--DeepSeek-V3.xlsx
Excel file saved as: C__205--mixtral-8x22b.xlsx
Excel file saved as: C__17--mixtral-8x22b.xlsx
Excel file saved as: C__48--mixtral-8x22b.xlsx
Excel file saved as: C__334--mixtral-8x22b.xlsx
Excel file saved as: C__267--llama-3-3-70b.xlsx
Excel file saved as: C__330--llama-3-3-70b.xlsx
Excel file saved as: C__227--DeepSeek-V3.xlsx
Excel file saved as: C__90--gpt-oss-120b.xlsx
Excel file saved as: C__86--mixtral-8x22b.xlsx
Excel file saved as: C__355--DeepSeek-V3.xlsx
Excel file saved as: C__210--llama-3-3-70b.xlsx
Excel file

In [8]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

def create_dpr_excel(dpr_text, gt_tables_metadata, output_filename='dpr_ground_truth.xlsx'):
    """
    Create an Excel file with DPR text and ground truth tables.
    
    Args:
        dpr_text: The DPR text string
        gt_tables_metadata: List of [table_id, table_title, table_headers]
        output_filename: Name of the output Excel file
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "DPR Ground Truth"
    
    # Styling
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF")

    dv = DataValidation(type="list", formula1='"Yes,No"', allow_blank=False)
    ws.add_data_validation(dv)
    
    # Write DPR text
    ws.merge_cells('A1:C1')
    ws['A1'] = "Data Product Request (DPR):"
    ws['A1'].font = Font(bold=True, size=12)

    ws.merge_cells('A2:D2')
    ws['A2'] = dpr_text
    ws['A2'].alignment = Alignment(wrap_text=True, vertical='top')
    ws.row_dimensions[2].height = 60  # Adjust height for DPR text

    ws['A3'] = "# of GT tables:"
    ws['B3'] = len(gt_tables_metadata)

    # Add spacing
    current_row = 5
    
    # Write Ground Truth header
    ws[f'A{current_row}'] = "DPR Only Annotations:"
    ws[f'A{current_row}'].font = Font(bold=True, size=16)
    current_row += 1

    # Write column headers
    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "Criteria"
    ws[f'D{current_row}'] = "Annotation (Yes/No)"
    for col in ['A', 'B', 'C', 'D']:
        ws[f'{col}{current_row}'].fill = header_fill
        ws[f'{col}{current_row}'].font = header_font
    current_row += 1
    
    # Write column headers
    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "Is the DPR high-quality and in the right level of abstraction?"
    ws[f'A{current_row}'].alignment = Alignment(wrap_text=True, vertical='top')
    dv.add(f'D{current_row}')
    current_row += 1

    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "Is the DPR clear i.e. is coherent, unambiguous, and easy to understand?"
    ws[f'A{current_row}'].alignment = Alignment(wrap_text=True, vertical='top')
    dv.add(f'D{current_row}')
    # Add spacing
    current_row += 2
    
    # Write Ground Truth header
    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "DPR - Ground Truth Alignments"
    ws[f'A{current_row}'].font = Font(bold=True, size=16)
    current_row += 1

    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "Please annotate if each of the following table is relevant to the given DPR or not."
    current_row += 2
    
    # Write column headers
    ws[f'A{current_row}'] = "Table ID"
    ws[f'B{current_row}'] = "Table Title"
    ws[f'C{current_row}'] = "Column Headers"
    ws[f'D{current_row}'] = "Relevant (Yes/No)"
    
    for col in ['A', 'B', 'C', 'D']:
        ws[f'{col}{current_row}'].fill = header_fill
        ws[f'{col}{current_row}'].font = header_font
    
    current_row += 1

    # Write ground truth tables (one per row)
    for table_metadata in gt_tables_metadata:
        table_id, table_title, table_headers = table_metadata
        
        ws[f'A{current_row}'] = table_id
        ws[f'B{current_row}'] = table_title
        ws[f'C{current_row}'] = ', '.join(table_headers)  # Join headers as comma-separated
        dv.add(f'D{current_row}')
        
        current_row += 1

    current_row += 1
    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "Criteria"
    ws[f'D{current_row}'] = "Annotation (Yes/No)"
    for col in ['A', 'B', 'C', 'D']:
        ws[f'{col}{current_row}'].fill = header_fill
        ws[f'{col}{current_row}'].font = header_font
    current_row += 1

    ws.merge_cells(f'A{current_row}:C{current_row}')
    ws[f'A{current_row}'] = "Does the scope of the set of ground truth tables and DPR roughly match? (i.e., scope of the DPR is not too broad with the possibility of many other valid tables not in GT)"
    ws[f'A{current_row}'].alignment = Alignment(wrap_text=True, vertical='top')
    dv.add(f'D{current_row}')
    current_row += 1
    

    
    # Adjust column widths
    ws.column_dimensions['A'].width = 40
    ws.column_dimensions['B'].width = 50
    ws.column_dimensions['C'].width = 60
    ws.column_dimensions['D'].width = 60
    
    # Save the workbook
    wb.save(output_filename)
    print(f"Excel file saved as: {output_filename}")